# ASR 模型評估 — SageMaker Notebook

**Instance**: ml.g5.xlarge (A10G 24 GB)  
**Kernel**: conda_pytorch

## Workflow

1. 安裝依賴 (faster-whisper, transformers)
2. 環境確認 (GPU / CUDA)
3. 載入測試語料 (`asr_test_utterances.jsonl`)
4. Taiwan-Tongues-CE 推論 (faster-whisper)
5. FormoSpeech Whisper-v3 推論 (transformers)
6. Roundtrip 測試：TTS 合成音檔 → ASR 辨識
7. 人工評分 (ipywidgets)
8. 匯出結果 CSV + 上傳 S3

## 測試模型

| Model | HuggingFace ID | Framework | 語言 |
|-------|----------------|-----------|------|
| Taiwan-Tongues-CE | `adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0` | faster-whisper | zh-TW |
| FormoSpeech Whisper-v3 | `formospeech/whisper-large-v3-taiwanese-hakka` | transformers | hak (六腔) |

## 評分維度 (1-5)

| 維度 | 5 | 3 | 1 |
|------|---|---|---|
| Completeness | 所有關鍵字命中 | 漏 1-2 個 | 多數遺漏 |
| Accuracy | 語意完全正確 | 有錯但意思保留 | 語意錯誤 |
| Usability | NLU 可直接使用 | 需容錯處理 | 無法使用 |

## 0. 安裝依賴

In [18]:
!pip install -q faster-whisper transformers accelerate soundfile librosa ipywidgets pandas

## 1. 環境確認

In [19]:
import torch
import json
import os
import time
import pandas as pd
import numpy as np
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display, Markdown

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

assert torch.cuda.is_available(), "需要 GPU 才能執行評估"

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A10G
VRAM: 22.1 GB


In [20]:
# 路徑設定
BASE_DIR = Path.home() / "SageMaker" / "eval"
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output" / "asr"
TTS_OUTPUT_DIR = BASE_DIR / "output"  # TTS 輸出（roundtrip 用）

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "taiwan_tongues_ce").mkdir(exist_ok=True)
(OUTPUT_DIR / "formospeech_whisper_v3").mkdir(exist_ok=True)
(OUTPUT_DIR / "roundtrip").mkdir(exist_ok=True)

CORPUS_FILE = INPUT_DIR / "asr_test_utterances.jsonl"

# 若語料不存在，從 S3 下載
if not CORPUS_FILE.exists():
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("語料不存在，從 S3 下載...")
    !aws s3 cp s3://e-hakka-care-eval-437814057855/input/ {INPUT_DIR}/ --recursive
    
assert CORPUS_FILE.exists(), f"語料不存在: {CORPUS_FILE}\n請手動執行: !aws s3 cp s3://e-hakka-care-eval-437814057855/input/ {INPUT_DIR}/ --recursive"

print(f"語料: {CORPUS_FILE}")
print(f"輸出: {OUTPUT_DIR}")

語料: /home/ec2-user/SageMaker/eval/input/asr_test_utterances.jsonl
輸出: /home/ec2-user/SageMaker/eval/output/asr


## 2. 載入測試語料

In [21]:
with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    utterances = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(utterances)
df_ce = df[df["model"] == "taiwan_tongues_ce"].reset_index(drop=True)
df_formo = df[df["model"] == "formospeech_whisper_v3"].reset_index(drop=True)

print(f"總計: {len(df)} 筆")
print(f"  Taiwan-Tongues-CE (zh-TW): {len(df_ce)} 筆")
print(f"  FormoSpeech Whisper-v3 (hak): {len(df_formo)} 筆")
display(df.head())

總計: 29 筆
  Taiwan-Tongues-CE (zh-TW): 25 筆
  FormoSpeech Whisper-v3 (hak): 4 筆


,id,model,lang,source,scenario,text,expected_keywords,difficulty,hakka_dialect
0,asr_ce_001,taiwan_tongues_ce,zh-TW,demo-script,服藥確認,血壓藥我吃了啦，剛剛配溫水吃的。,"[血壓藥, 吃了, 溫水]",easy,NaN
1,asr_ce_002,taiwan_tongues_ce,zh-TW,demo-script,血壓數值,有量，高的一百三十幾，低的八十。,"[一百三十, 八十]",medium,NaN
2,asr_ce_003,taiwan_tongues_ce,zh-TW,demo-script,膝蓋疼痛,我膝蓋有點痛，走路怕跌倒。,"[膝蓋, 痛, 跌倒]",easy,NaN
3,asr_ce_004,taiwan_tongues_ce,zh-TW,demo-script,浴室滑倒,其實我今天早上在浴室滑了一下，還好有扶到牆壁。,"[浴室, 滑, 牆壁]",medium,NaN
4,asr_ce_005,taiwan_tongues_ce,zh-TW,elder_001,晨間問候,早安，今天好像有一點冷。,"[早安, 冷]",easy,NaN


## 3. 準備音檔

ASR 需要輸入音檔。此處提供兩種來源：
1. **預錄音檔**：如有真人錄音放在 `corpus/audio/` 目錄
2. **TTS 合成音檔**：從前一步 TTS 評估產出的 WAV 檔（roundtrip 測試）

若無預錄音檔，我們使用 gTTS 或 edge-tts 合成簡易測試音檔。

In [22]:
# 檢查 TTS 輸出是否存在（roundtrip 用）
tts_voxhakka_dir = TTS_OUTPUT_DIR / "voxhakka"
tts_omnivoice_dir = TTS_OUTPUT_DIR / "omnivoice"

tts_wavs_available = tts_voxhakka_dir.exists() and any(tts_voxhakka_dir.glob("*.wav"))
print(f"TTS VoxHakka 音檔可用: {tts_wavs_available}")

if tts_wavs_available:
    vox_wavs = sorted(tts_voxhakka_dir.glob("*.wav"))
    omni_wavs = sorted(tts_omnivoice_dir.glob("*.wav")) if tts_omnivoice_dir.exists() else []
    print(f"  VoxHakka WAVs: {len(vox_wavs)}")
    print(f"  OmniVoice WAVs: {len(omni_wavs)}")
else:
    print("\n若需 roundtrip 測試，請先執行 TTS 評估 notebook 或從 S3 下載：")
    print(f"  !aws s3 cp s3://e-hakka-care-eval-437814057855/output/tts/ {TTS_OUTPUT_DIR}/ --recursive")

TTS VoxHakka 音檔可用: True
  VoxHakka WAVs: 7
  OmniVoice WAVs: 5


In [23]:
# 為 zh-TW 測試語料合成簡易測試音檔（edge-tts）
# 這提供一個 baseline 音源讓 ASR 有東西可辨識
!pip install -q edge-tts

import edge_tts

SYNTH_DIR = OUTPUT_DIR / "synth_input"
SYNTH_DIR.mkdir(exist_ok=True)

async def synthesize_test_audio(row):
    out_path = SYNTH_DIR / f"{row['id']}.wav"
    if out_path.exists():
        return str(out_path)
    communicate = edge_tts.Communicate(row["text"], "zh-TW-HsiaoChenNeural")
    await communicate.save(str(out_path))
    return str(out_path)

# 合成 CE 測試音檔
print("合成 zh-TW 測試音檔...")
for _, row in df_ce.iterrows():
    path = await synthesize_test_audio(row)
    
synth_wavs = sorted(SYNTH_DIR.glob("*.wav"))
print(f"已合成 {len(synth_wavs)} 個測試音檔")

合成 zh-TW 測試音檔...
已合成 25 個測試音檔


## 4. Taiwan-Tongues-CE 推論 (faster-whisper)

Model: `adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0`  
Framework: faster-whisper (CTranslate2)  
語言: zh-TW

In [24]:
from faster_whisper import WhisperModel

print("載入 Taiwan-Tongues-CE...")
t0 = time.time()
ce_model = WhisperModel(
    "adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0",
    device="cuda",
    compute_type="float16"
)
print(f"載入完成 ({time.time() - t0:.1f}s)")

載入 Taiwan-Tongues-CE...
載入完成 (22.6s)


In [25]:
# 辨識合成音檔
import warnings
import logging

# 抑制 ONNX Runtime 的 sysfs warning（容器環境無 /sys/class/drm）
logging.getLogger("faster_whisper").setLevel(logging.ERROR)

ce_results = []

for _, row in df_ce.iterrows():
    audio_path = SYNTH_DIR / f"{row['id']}.wav"
    if not audio_path.exists():
        print(f"  [SKIP] {row['id']} - 音檔不存在")
        continue
    
    t0 = time.time()
    segments, info = ce_model.transcribe(
        str(audio_path),
        language="zh",
        beam_size=5,
        vad_filter=True
    )
    transcript = "".join(seg.text for seg in segments)
    elapsed = time.time() - t0
    
    # 關鍵字命中檢查
    keywords = row["expected_keywords"]
    hits = [kw for kw in keywords if kw in transcript]
    
    result = {
        "id": row["id"],
        "reference": row["text"],
        "transcript": transcript,
        "keywords": keywords,
        "keyword_hits": hits,
        "keyword_hit_rate": len(hits) / len(keywords) if keywords else 0,
        "scenario": row["scenario"],
        "difficulty": row["difficulty"],
        "latency_s": elapsed,
        "detected_lang": info.language,
        "lang_prob": info.language_probability
    }
    ce_results.append(result)
    print(f"  [{row['id']}] {elapsed:.2f}s | hits={len(hits)}/{len(keywords)} | {transcript[:40]}...")

df_ce_results = pd.DataFrame(ce_results)
print(f"\n完成: {len(ce_results)}/{len(df_ce)} 筆")
print(f"平均關鍵字命中率: {df_ce_results['keyword_hit_rate'].mean():.2%}")

  [asr_ce_001] 0.78s | hits=3/3 | 血壓藥我吃了啦剛剛配溫水吃的...
  [asr_ce_002] 0.57s | hits=2/2 | 有量高的一百三十幾低的八十...
  [asr_ce_003] 0.56s | hits=3/3 | 我膝蓋有點痛走路怕跌倒...
  [asr_ce_004] 0.67s | hits=3/3 | 其實我今天早上在浴室滑了一下還好有浮到牆壁...
  [asr_ce_005] 0.50s | hits=2/2 | 早安今天好像有點冷...
  [asr_ce_006] 0.59s | hits=4/4 | 還可以只是昨晚睡得不太好半夜醒來兩次...
  [asr_ce_007] 0.55s | hits=3/3 | 現在幾點下午是不是要去看醫生...
  [asr_ce_008] 0.56s | hits=2/3 | 兩點三十還是一點三十我怕計錯...
  [asr_ce_009] 0.58s | hits=3/3 | 好我看到了早餐後一顆白色的藥...
  [asr_ce_010] 0.49s | hits=2/2 | 我吃好了沒有不舒服...
  [asr_ce_011] 0.62s | hits=3/3 | 今天台北會不會下雨我想去附近的公園走走...
  [asr_ce_012] 0.55s | hits=2/2 | 你說什麼剛剛車子開過去我沒有聽清楚 ...
  [asr_ce_013] 0.53s | hits=1/2 | 曉琳今天會來看我嗎...
  [asr_ce_014] 0.56s | hits=2/2 | 今天心裡悶悶的也不知道要跟誰說...
  [asr_ce_015] 0.58s | hits=3/3 | 我想起以前和先生一起種花的日子有點想念他...
  [asr_ce_016] 0.56s | hits=3/3 | 不要提醒我喝咖啡我要喝溫水...
  [asr_ce_017] 0.53s | hits=3/3 | 不用我等一下吃完點心再喝...
  [asr_ce_018] 0.90s | hits=4/6 | 明天早上我要先去市場買雞蛋青菜和豆腐回來以後再晒衣服下午如果沒有下雨想請女兒陪我...
  [asr_ce_019] 0.61s | hits=3/3 |

In [26]:
# 檢視結果
display(df_ce_results[["id", "reference", "transcript", "keyword_hit_rate", "latency_s"]].head(10))

,id,reference,transcript,keyword_hit_rate,latency_s
0,asr_ce_001,血壓藥我吃了啦，剛剛配溫水吃的。,血壓藥我吃了啦剛剛配溫水吃的,1.000000,0.778093
1,asr_ce_002,有量，高的一百三十幾，低的八十。,有量高的一百三十幾低的八十,1.000000,0.570006
2,asr_ce_003,我膝蓋有點痛，走路怕跌倒。,我膝蓋有點痛走路怕跌倒,1.000000,0.558416
3,asr_ce_004,其實我今天早上在浴室滑了一下，還好有扶到牆壁。,其實我今天早上在浴室滑了一下還好有浮到牆壁,1.000000,0.665895
4,asr_ce_005,早安，今天好像有一點冷。,早安今天好像有點冷,1.000000,0.499081
5,asr_ce_006,還可以，只是昨晚睡得不太好，半夜醒來兩次。,還可以只是昨晚睡得不太好半夜醒來兩次,1.000000,0.590557
6,asr_ce_007,現在幾點？下午是不是要去看醫生？,現在幾點下午是不是要去看醫生,1.000000,0.551515
7,asr_ce_008,兩點三十，還是一點三十？我怕記錯。,兩點三十還是一點三十我怕計錯,0.666667,0.564867
8,asr_ce_009,好，我看到了，早餐後一顆白色的藥。,好我看到了早餐後一顆白色的藥,1.000000,0.576173
9,asr_ce_010,我吃好了，沒有不舒服。,我吃好了沒有不舒服,1.000000,0.494482


In [27]:
# 釋放 GPU 記憶體
del ce_model
torch.cuda.empty_cache()
print("Taiwan-Tongues-CE 模型已卸載")

Taiwan-Tongues-CE 模型已卸載


## 5. FormoSpeech Whisper-v3 推論 (transformers)

Model: `formospeech/whisper-large-v3-taiwanese-hakka`  
Framework: transformers  
語言: hak (客語六腔)

In [28]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import librosa

print("載入 FormoSpeech Whisper-v3...")
t0 = time.time()

formo_processor = WhisperProcessor.from_pretrained(
    "formospeech/whisper-large-v3-taiwanese-hakka"
)
formo_model = WhisperForConditionalGeneration.from_pretrained(
    "formospeech/whisper-large-v3-taiwanese-hakka",
    torch_dtype=torch.float16
).to("cuda")

print(f"載入完成 ({time.time() - t0:.1f}s)")

載入 FormoSpeech Whisper-v3...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

載入完成 (28.8s)


In [29]:
# FormoSpeech 需要客語音檔輸入
# 使用 TTS 評估產出的 VoxHakka 音檔作為輸入
import warnings
warnings.filterwarnings("ignore", message=".*forced_decoder_ids.*")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*and.*max_length.*")
warnings.filterwarnings("ignore", message=".*SuppressTokens.*")
warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")

formo_results = []

# 建立 TTS 語料的 text lookup（roundtrip 驗證用）
tts_corpus_file = INPUT_DIR / "tts_test_utterances.jsonl"
tts_lookup = {}
if tts_corpus_file.exists():
    with open(tts_corpus_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                tts_lookup[item["id"]] = item

# 辨識 VoxHakka TTS 輸出
if tts_wavs_available:
    for wav_path in vox_wavs:
        item_id = wav_path.stem
        ref_item = tts_lookup.get(item_id, {})
        ref_text = ref_item.get("text", "")
        
        t0 = time.time()
        audio, sr = librosa.load(str(wav_path), sr=16000)
        input_features = formo_processor(
            audio, sampling_rate=16000, return_tensors="pt"
        ).input_features.to("cuda", dtype=torch.float16)
        
        forced_decoder_ids = formo_processor.get_decoder_prompt_ids(
            language="chinese", task="transcribe"
        )
        
        with torch.no_grad():
            predicted_ids = formo_model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=256
            )
        
        transcript = formo_processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0]
        elapsed = time.time() - t0
        
        result = {
            "id": item_id,
            "source": "voxhakka_tts",
            "reference": ref_text,
            "transcript": transcript,
            "scenario": ref_item.get("scenario", ""),
            "hakka_dialect": ref_item.get("hakka_dialect", ""),
            "latency_s": elapsed
        }
        formo_results.append(result)
        print(f"  [{item_id}] {elapsed:.2f}s | {transcript[:40]}...")
else:
    print("無 TTS 音檔可用。請先執行 TTS 評估或從 S3 下載音檔。")
    print(f"  !aws s3 cp s3://e-hakka-care-eval-437814057855/output/tts/voxhakka/ {TTS_OUTPUT_DIR}/voxhakka/ --recursive")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [tts_vox_001] 0.66s | 食飽吔就餓想今晡日有去菜園無...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [tts_vox_002] 0.53s | 呵𠊎會講慢兜喚你慢慢了...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [tts_vox_004] 0.83s | 泥泥八點半到咧果汁藥仔咧愛呸暖水食好...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [tts_vox_005] 0.78s | 𠊎在這位喚你慢慢講毋使急想講麼个就講麼个...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [tts_vox_006] 0.50s | 食飽吔今晡日天時蓋好...
  [tts_vox_007] 0.48s | 食飽咧今晡日天時當好...


In [30]:
# 也辨識 ASR 語料中的 Formo 測試句（asr_formo_001~004）
# 這些句子需要客語音檔，嘗試從 VoxHakka TTS 輸出中以文本匹配找到對應音檔
tts_text_to_wav = {}
if tts_wavs_available:
    for wav_path in vox_wavs:
        item_id = wav_path.stem
        ref_item = tts_lookup.get(item_id, {})
        if ref_item.get("text"):
            tts_text_to_wav[ref_item["text"]] = wav_path

for _, row in df_formo.iterrows():
    # 嘗試多種來源
    audio_path = None
    source_label = ""
    
    # 1. 直接 ID 匹配
    for candidate in [
        SYNTH_DIR / f"{row['id']}.wav",
        INPUT_DIR / "audio" / f"{row['id']}.wav",
    ]:
        if candidate.exists():
            audio_path = candidate
            source_label = "direct_match"
            break
    
    # 2. 透過文本匹配 VoxHakka TTS 輸出
    if audio_path is None and row["text"] in tts_text_to_wav:
        audio_path = tts_text_to_wav[row["text"]]
        source_label = "voxhakka_text_match"
    
    if audio_path is None:
        print(f"  [SKIP] {row['id']} - 無音檔（文本: {row['text'][:20]}...）")
        continue
    
    t0 = time.time()
    audio, sr = librosa.load(str(audio_path), sr=16000)
    input_features = formo_processor(
        audio, sampling_rate=16000, return_tensors="pt"
    ).input_features.to("cuda", dtype=torch.float16)
    
    forced_decoder_ids = formo_processor.get_decoder_prompt_ids(
        language="chinese", task="transcribe"
    )
    
    with torch.no_grad():
        predicted_ids = formo_model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=256
        )
    
    transcript = formo_processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]
    elapsed = time.time() - t0
    
    keywords = row["expected_keywords"]
    hits = [kw for kw in keywords if kw in transcript]
    
    result = {
        "id": row["id"],
        "source": source_label,
        "reference": row["text"],
        "transcript": transcript,
        "keywords": keywords,
        "keyword_hits": hits,
        "keyword_hit_rate": len(hits) / len(keywords) if keywords else 0,
        "scenario": row["scenario"],
        "hakka_dialect": row.get("hakka_dialect", ""),
        "latency_s": elapsed
    }
    formo_results.append(result)
    print(f"  [{row['id']}] ({source_label}) {elapsed:.2f}s | hits={len(hits)}/{len(keywords)} | {transcript[:40]}...")

df_formo_results = pd.DataFrame(formo_results) if formo_results else pd.DataFrame()
print(f"\nFormoSpeech 完成: {len(formo_results)} 筆")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [SKIP] asr_formo_001 - 無音檔（文本: 今晡日食飽未？...）
  [SKIP] asr_formo_002 - 無音檔（文本: 𠊎今晡日有兜仔想食粥。...）
  [SKIP] asr_formo_003 - 無音檔（文本: 好啊，毋過你講慢兜，我較聽得清楚。...）
  [asr_formo_004] (voxhakka_text_match) 0.64s | hits=2/3 | 食飽吔就餓想今晡日有去菜園無...

FormoSpeech 完成: 8 筆


In [31]:
if not df_formo_results.empty:
    display(df_formo_results[["id", "source", "reference", "transcript", "latency_s"]].head(10))

,id,source,reference,transcript,latency_s
0,tts_vox_001,voxhakka_tts,食飽咧，秋妹嬸今晡日有去菜園無？,食飽吔就餓想今晡日有去菜園無,0.657390
1,tts_vox_002,voxhakka_tts,好，𠊎會講慢兜，陪您慢慢聊。,呵𠊎會講慢兜喚你慢慢了,0.533787
2,tts_vox_003,voxhakka_tts,陳奶奶，早安！今晡日天光真好。昨暗晡睡得好無？,塵泥泥早安今晡日天光真好昨暗晡睡得好無,0.811161
3,tts_vox_004,voxhakka_tts,奶奶，八點半到咧！該食藥仔咧，愛配暖水食喔。,泥泥八點半到咧果汁藥仔咧愛呸暖水食好,0.833834
4,tts_vox_005,voxhakka_tts,𠊎在這位，陪您慢慢講。毋使急，想講麼个就講麼个。,𠊎在這位喚你慢慢講毋使急想講麼个就講麼个,0.777578
5,tts_vox_006,voxhakka_tts,食飽咧，今晡日天時蓋好。,食飽吔今晡日天時蓋好,0.504810
6,tts_vox_007,voxhakka_tts,食飽咧，今晡日天時當好。,食飽咧今晡日天時當好,0.475328
7,asr_formo_004,voxhakka_text_match,食飽咧，秋妹嬸今晡日有去菜園無？,食飽吔就餓想今晡日有去菜園無,0.640128


In [32]:
# 釋放 GPU 記憶體
del formo_model, formo_processor
torch.cuda.empty_cache()
print("FormoSpeech Whisper-v3 模型已卸載")

FormoSpeech Whisper-v3 模型已卸載


## 6. Roundtrip 測試：TTS → ASR

用 Taiwan-Tongues-CE 辨識 TTS 產出的客語音檔，模擬實際 pipeline：  
長者說客語 → TTS 合成 → ASR 辨識 → NLU 處理

此測試驗證 TTS + ASR 端到端的相容性。

In [33]:
roundtrip_results = []

if tts_wavs_available:
    # 重新載入 CE 模型做 roundtrip
    print("載入 Taiwan-Tongues-CE (roundtrip)...")
    ce_model_rt = WhisperModel(
        "adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0",
        device="cuda",
        compute_type="float16"
    )
    
    all_tts_wavs = vox_wavs + omni_wavs
    for wav_path in all_tts_wavs:
        item_id = wav_path.stem
        tts_source = "voxhakka" if "vox" in item_id else "omnivoice"
        ref_item = tts_lookup.get(item_id, {})
        ref_text = ref_item.get("text", "")
        
        t0 = time.time()
        segments, info = ce_model_rt.transcribe(
            str(wav_path),
            language="zh",
            beam_size=5,
            vad_filter=True
        )
        transcript = "".join(seg.text for seg in segments)
        elapsed = time.time() - t0
        
        result = {
            "id": item_id,
            "tts_source": tts_source,
            "tts_text": ref_text,
            "asr_transcript": transcript,
            "scenario": ref_item.get("scenario", ""),
            "latency_s": elapsed
        }
        roundtrip_results.append(result)
        print(f"  [{item_id}] {tts_source} → CE | {elapsed:.2f}s | {transcript[:40]}...")
    
    del ce_model_rt
    torch.cuda.empty_cache()
    
    df_roundtrip = pd.DataFrame(roundtrip_results)
    print(f"\nRoundtrip 完成: {len(roundtrip_results)} 筆")
else:
    df_roundtrip = pd.DataFrame()
    print("跳過 roundtrip 測試（無 TTS 音檔）")

載入 Taiwan-Tongues-CE (roundtrip)...
  [tts_vox_001] voxhakka → CE | 0.68s | 吃飽了嫦娥吃今天有去菜園嗎...
  [tts_vox_002] voxhakka → CE | 0.55s | 好你再說慢些讓你慢慢留 ...
  [tts_vox_003] voxhakka → CE | 0.61s | 陳奶奶早上今天天光正好坐晚上睡得好嗎...
  [tts_vox_004] voxhakka → CE | 0.64s | 乖乖八點半到了高市藥物要放暖水吃好 ...
  [tts_vox_005] voxhakka → CE | 0.61s | 我找他的位置讓你慢慢說不用急想說什麼就說什麼 ...
  [tts_vox_006] voxhakka → CE | 0.51s | 吃飽了今天天氣很好...
  [tts_vox_007] voxhakka → CE | 0.53s | 吃飽了今天天氣很好 ...
  [tts_omni_001] omnivoice → CE | 0.58s | 吃飽了就不要嫌今天有去菜園嗎...
  [tts_omni_002] omnivoice → CE | 0.54s | 好我會說慢些陪人慢慢聊...
  [tts_omni_003] omnivoice → CE | 0.61s | 陳奶奶早安今天天氣真好坐晚上睡得好嗎...
  [tts_omni_004] omnivoice → CE | 0.63s | 糟糕八點半到了那吃藥要配暖水吃喔...
  [tts_omni_005] omnivoice → CE | 0.62s | 我在這裡為了陪人慢慢說不用急想說什麼就說什麼 ...

Roundtrip 完成: 12 筆


In [34]:
if not df_roundtrip.empty:
    display(df_roundtrip[["id", "tts_source", "tts_text", "asr_transcript", "latency_s"]])

,id,tts_source,tts_text,asr_transcript,latency_s
0,tts_vox_001,voxhakka,食飽咧，秋妹嬸今晡日有去菜園無？,吃飽了嫦娥吃今天有去菜園嗎,0.684639
1,tts_vox_002,voxhakka,好，𠊎會講慢兜，陪您慢慢聊。,好你再說慢些讓你慢慢留,0.554153
2,tts_vox_003,voxhakka,陳奶奶，早安！今晡日天光真好。昨暗晡睡得好無？,陳奶奶早上今天天光正好坐晚上睡得好嗎,0.607536
3,tts_vox_004,voxhakka,奶奶，八點半到咧！該食藥仔咧，愛配暖水食喔。,乖乖八點半到了高市藥物要放暖水吃好,0.638696
4,tts_vox_005,voxhakka,𠊎在這位，陪您慢慢講。毋使急，想講麼个就講麼个。,我找他的位置讓你慢慢說不用急想說什麼就說什麼,0.613553
5,tts_vox_006,voxhakka,食飽咧，今晡日天時蓋好。,吃飽了今天天氣很好,0.507993
6,tts_vox_007,voxhakka,食飽咧，今晡日天時當好。,吃飽了今天天氣很好,0.527963
7,tts_omni_001,omnivoice,食飽咧，秋妹嬸今晡日有去菜園無？,吃飽了就不要嫌今天有去菜園嗎,0.579444
8,tts_omni_002,omnivoice,好，𠊎會講慢兜，陪您慢慢聊。,好我會說慢些陪人慢慢聊,0.541836
9,tts_omni_003,omnivoice,陳奶奶，早安！今晡日天光真好。昨暗晡睡得好無？,陳奶奶早安今天天氣真好坐晚上睡得好嗎,0.607005


## 7. 人工評分

評分維度（1-5）：
- **Completeness**：關鍵字命中完整度
- **Accuracy**：語意正確度
- **Usability**：NLU 可用度

In [35]:
import ipywidgets as widgets

# 合併所有結果供評分
all_results = []

for r in ce_results:
    all_results.append({
        "id": r["id"],
        "model": "taiwan_tongues_ce",
        "source_type": "synth_input",
        "reference": r["reference"],
        "transcript": r["transcript"],
        "scenario": r["scenario"]
    })

for r in formo_results:
    all_results.append({
        "id": r["id"],
        "model": "formospeech_whisper_v3",
        "source_type": r.get("source", "tts"),
        "reference": r["reference"],
        "transcript": r["transcript"],
        "scenario": r["scenario"]
    })

for r in roundtrip_results:
    all_results.append({
        "id": r["id"],
        "model": "roundtrip_ce",
        "source_type": r["tts_source"],
        "reference": r["tts_text"],
        "transcript": r["asr_transcript"],
        "scenario": r["scenario"]
    })

print(f"待評分: {len(all_results)} 筆")

# 評分收集
scores = []
current_idx = [0]

def show_item(idx):
    if idx >= len(all_results):
        print("\n✓ 所有項目已評分完成！")
        return
    item = all_results[idx]
    display(Markdown(f"### [{idx+1}/{len(all_results)}] {item['id']} ({item['model']})"))
    display(Markdown(f"**情境**: {item['scenario']}"))
    display(Markdown(f"**原文**: {item['reference']}"))
    display(Markdown(f"**辨識**: {item['transcript']}"))

completeness = widgets.IntSlider(value=3, min=1, max=5, description="Completeness:")
accuracy = widgets.IntSlider(value=3, min=1, max=5, description="Accuracy:")
usability = widgets.IntSlider(value=3, min=1, max=5, description="Usability:")
notes = widgets.Text(value="", description="備註:", layout=widgets.Layout(width="80%"))

def on_submit(btn):
    item = all_results[current_idx[0]]
    scores.append({
        "id": item["id"],
        "model": item["model"],
        "source_type": item["source_type"],
        "completeness": completeness.value,
        "accuracy": accuracy.value,
        "usability": usability.value,
        "notes": notes.value
    })
    completeness.value = 3
    accuracy.value = 3
    usability.value = 3
    notes.value = ""
    current_idx[0] += 1
    from IPython.display import clear_output
    clear_output(wait=True)
    display(widgets.VBox([completeness, accuracy, usability, notes, submit_btn]))
    show_item(current_idx[0])

submit_btn = widgets.Button(description="下一筆 →", button_style="primary")
submit_btn.on_click(on_submit)

display(widgets.VBox([completeness, accuracy, usability, notes, submit_btn]))
show_item(0)


✓ 所有項目已評分完成！


## 8. 匯出結果

In [40]:
# 匯出評分 CSV
if scores:
    df_scores = pd.DataFrame(scores)
    
    # 分模型匯出
    for model_name in df_scores["model"].unique():
        model_df = df_scores[df_scores["model"] == model_name]
        csv_path = OUTPUT_DIR / f"asr_eval_{model_name}.csv"
        model_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"已匯出: {csv_path}")
        
        # 統計
        dims = ["completeness", "accuracy", "usability"]
        print(f"  {model_name}:")
        for d in dims:
            print(f"    {d}: mean={model_df[d].mean():.2f}, min={model_df[d].min()}, max={model_df[d].max()}")
        mos = model_df[dims].mean().mean()
        print(f"    Overall MOS: {mos:.2f}")
        print()
else:
    print("尚未完成評分，請先執行上方評分 widget")

已匯出: /home/ec2-user/SageMaker/eval/output/asr/asr_eval_taiwan_tongues_ce.csv
  taiwan_tongues_ce:
    completeness: mean=4.68, min=3, max=5
    accuracy: mean=4.68, min=2, max=5
    usability: mean=4.64, min=2, max=5
    Overall MOS: 4.67

已匯出: /home/ec2-user/SageMaker/eval/output/asr/asr_eval_formospeech_whisper_v3.csv
  formospeech_whisper_v3:
    completeness: mean=3.75, min=2, max=5
    accuracy: mean=3.00, min=1, max=5
    usability: mean=2.88, min=1, max=5
    Overall MOS: 3.21

已匯出: /home/ec2-user/SageMaker/eval/output/asr/asr_eval_roundtrip_ce.csv
  roundtrip_ce:
    completeness: mean=4.08, min=2, max=5
    accuracy: mean=3.08, min=2, max=5
    usability: mean=3.00, min=1, max=5
    Overall MOS: 3.39



In [41]:
# 匯出辨識結果 CSV（不含人工評分，純機器指標）
if ce_results:
    df_ce_results.to_csv(
        OUTPUT_DIR / "asr_transcripts_taiwan_tongues_ce.csv",
        index=False, encoding="utf-8-sig"
    )
    print(f"CE 辨識結果: {OUTPUT_DIR / 'asr_transcripts_taiwan_tongues_ce.csv'}")

if formo_results:
    df_formo_results.to_csv(
        OUTPUT_DIR / "asr_transcripts_formospeech.csv",
        index=False, encoding="utf-8-sig"
    )
    print(f"Formo 辨識結果: {OUTPUT_DIR / 'asr_transcripts_formospeech.csv'}")

if roundtrip_results:
    df_roundtrip.to_csv(
        OUTPUT_DIR / "asr_roundtrip_results.csv",
        index=False, encoding="utf-8-sig"
    )
    print(f"Roundtrip 結果: {OUTPUT_DIR / 'asr_roundtrip_results.csv'}")

CE 辨識結果: /home/ec2-user/SageMaker/eval/output/asr/asr_transcripts_taiwan_tongues_ce.csv
Formo 辨識結果: /home/ec2-user/SageMaker/eval/output/asr/asr_transcripts_formospeech.csv
Roundtrip 結果: /home/ec2-user/SageMaker/eval/output/asr/asr_roundtrip_results.csv


In [42]:
# 彙整報告 JSON
report = {
    "instance": "ml.g5.xlarge",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "models": {
        "taiwan_tongues_ce": {
            "hf_id": "adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0",
            "framework": "faster-whisper",
            "language": "zh-TW",
            "total_utterances": len(ce_results),
            "avg_keyword_hit_rate": float(df_ce_results["keyword_hit_rate"].mean()) if ce_results else None,
            "avg_latency_s": float(df_ce_results["latency_s"].mean()) if ce_results else None
        },
        "formospeech_whisper_v3": {
            "hf_id": "formospeech/whisper-large-v3-taiwanese-hakka",
            "framework": "transformers",
            "language": "hak",
            "total_utterances": len(formo_results),
            "avg_latency_s": float(df_formo_results["latency_s"].mean()) if formo_results else None
        }
    },
    "roundtrip": {
        "total_utterances": len(roundtrip_results),
        "tts_sources": list(df_roundtrip["tts_source"].unique()) if not df_roundtrip.empty else []
    }
}

report_path = OUTPUT_DIR / "asr_eval_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"報告: {report_path}")
print(json.dumps(report, ensure_ascii=False, indent=2))

報告: /home/ec2-user/SageMaker/eval/output/asr/asr_eval_report.json
{
  "instance": "ml.g5.xlarge",
  "gpu": "NVIDIA A10G",
  "models": {
    "taiwan_tongues_ce": {
      "hf_id": "adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0",
      "framework": "faster-whisper",
      "language": "zh-TW",
      "total_utterances": 25,
      "avg_keyword_hit_rate": 0.9033333333333333,
      "avg_latency_s": 0.6185442543029785
    },
    "formospeech_whisper_v3": {
      "hf_id": "formospeech/whisper-large-v3-taiwanese-hakka",
      "framework": "transformers",
      "language": "hak",
      "total_utterances": 8,
      "avg_latency_s": 0.6542518436908722
    }
  },
  "roundtrip": {
    "total_utterances": 12,
    "tts_sources": [
      "voxhakka",
      "omnivoice"
    ]
  }
}


## 9. 上傳 S3

In [39]:
S3_BUCKET = "s3://e-hakka-care-eval-437814057855"
S3_PREFIX = f"{S3_BUCKET}/output/asr/"

!aws s3 cp {OUTPUT_DIR}/ {S3_PREFIX} --recursive --exclude "synth_input/*"
print(f"\n已上傳至: {S3_PREFIX}")
!aws s3 ls {S3_PREFIX} --recursive

upload: output/asr/asr_roundtrip_results.csv to s3://e-hakka-care-eval-437814057855/output/asr/asr_roundtrip_results.csv
upload: output/asr/asr_eval_report.json to s3://e-hakka-care-eval-437814057855/output/asr/asr_eval_report.json
upload: output/asr/asr_transcripts_formospeech.csv to s3://e-hakka-care-eval-437814057855/output/asr/asr_transcripts_formospeech.csv
upload: output/asr/asr_transcripts_taiwan_tongues_ce.csv to s3://e-hakka-care-eval-437814057855/output/asr/asr_transcripts_taiwan_tongues_ce.csv

已上傳至: s3://e-hakka-care-eval-437814057855/output/asr/
2026-08-01 15:16:56        696 output/asr/asr_eval_report.json
2026-08-01 15:16:56       2011 output/asr/asr_roundtrip_results.csv
2026-08-01 15:16:56       1544 output/asr/asr_transcripts_formospeech.csv
2026-08-01 15:16:56       6273 output/asr/asr_transcripts_taiwan_tongues_ce.csv


## 10. 停止 Instance（省錢！）

評估完成後，記得停止 notebook instance：
```bash
aws sagemaker stop-notebook-instance --notebook-instance-name tts-asr-eval
```